<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/04-evals/01-golden-sets.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 黄金集

**目标：** 为第 03 节的 RAG 系统构建一个黄金集，并在其上运行廉价的确定性检查。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。


## 环境准备

每个 notebook 都是自包含的，所以接下来两个单元格会从零把它搭起来：

1. **安装依赖。** `aien` 包（就是本仓库）带有共享的环境准备助手，并会拉入 `groq` 客户端；本 notebook 还会用到 `sentence-transformers`、`numpy`。
2. **加载你的 API key。** 到 [console.groq.com](https://console.groq.com/) 领一个免费 key（无需信用卡）。在 Colab 里，通过左侧边栏的 **钥匙图标** → **Add new secret** 添加它，名称必须严格写成 `GROQ_API_KEY`，粘贴值，然后打开 **Notebook access** 开关。在本地运行？把 `GROQ_API_KEY` 设为环境变量。

（完整走查和选模型指引见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main" sentence-transformers numpy

In [ ]:
from aien import setup

# 读取 GROQ_API_KEY（Colab Secrets 或本地环境变量），并返回一个就绪的
# Groq 客户端。传 model=... 可覆盖默认值；如果之后某次调用报 404，
# 就列出可用模型 —— 见 00-setup/00-environment.ipynb。
client, MODEL = setup()

> **🔵 撞上速率限制？换模型。** 评测类 notebook 很费 token（每个 case 一次模型调用，跑完整个集合），而 Groq 免费档的每分钟 token 上限和每日 token 上限都是**按模型**计算的。如果你收到 `429`，把 `setup()` 指向另一个模型。它有独立的预算：
> ```python
> client, MODEL = setup(model='openai/gpt-oss-20b')     # 或者 'qwen/qwen3.6-27b'
> ```
> 从 [Groq 速率限制页](https://console.groq.com/docs/rate-limits) 挑一个还有余量的。只为*验证*跑通的话，任何有能力的对话模型都行；重点是方法论，不是精确的分数。这里避开 agentic 的 `groq/compound*` 模型：它们会浏览网页并运行代码，那会破坏「用落地事实作答」这一课。

## 为什么要有这个 notebook

评测是 AI 工程师能发出的最强「我真的交付过」信号。一个 demo 只说明系统能跑通一次；一份评测说明你知道它什么时候跑不通。任何人都能把检索接到模型上，在台上拿到一个好答案。真正被录用的工程师，是那个能说「我们在黄金集上是 87%，三个失败全是多跳问题，这是和上周的 diff」的人。

这也正是你在后端工作里早就有的直觉：每个生产 bug 都会变成一个回归测试。这里的做法一样。每个坏答案都会变成一个黄金集 case。

在这个 notebook 里，我们构建黄金集，并在其上跑两个确定性指标。接下来两个 notebook 会在此基础上加入 LLM 评判和回归测试骨架。


## 被测系统

接下来两个单元格以紧凑形式重建第 03 节的 RAG 系统：下载并清洗十份 IETF RFC，按章节分块，用一个小型本地模型做 embedding，按余弦相似度检索，并带引用作答。

这是一次刻意的复制，而不是 import。每个 notebook 都必须能在全新的 Colab 运行时里从上到下跑通，对教学材料来说，自包含比 DRY 更重要。如果这些细节你不熟，先过一遍 `03-rag/`。这里它们只是我们要评测的对象。


In [ ]:
# 03-rag pipeline 的紧凑版复制。自包含优先于 DRY：本 notebook 必须能
# 在 Colab 里独立跑通。这里每个设计决策的完整走查见 03-rag/。
import os
import re
import urllib.request

RFCS = [791, 793, 1035, 2616, 4271, 5321, 6455, 6749, 7540, 9110]
os.makedirs('data/rfc', exist_ok=True)
for n in RFCS:
    path = f'data/rfc/rfc{n}.txt'
    if not os.path.exists(path):  # 如果前面的 notebook 已经缓存过就跳过
        urllib.request.urlretrieve(f'https://www.rfc-editor.org/rfc/rfc{n}.txt', path)

def clean_rfc(text):
    """去掉换页符和 RFC txt 文件自带的页眉/页脚行。"""
    text = text.replace('\f', '\n')
    lines = [l for l in text.split('\n')
             if not re.search(r'\[Page \d+\]\s*$', l)          # 页脚
             and not re.match(r'^\s*RFC \d+.*\d{4}\s*$', l)]   # 页眉
    return re.sub(r'\n{3,}', '\n\n', '\n'.join(lines))

def chunk_rfc(text, rfc, max_chars=2000):
    """按章节分块：按带编号的标题切分，然后限制块的大小。"""
    parts = re.split(r'\n(?=\d+(?:\.\d+)*\.?\s+[A-Z])', text)
    chunks = []
    for part in parts:
        part = part.strip()
        while len(part) > max_chars:
            cut = part.rfind('\n\n', 0, max_chars)
            cut = cut if cut > 200 else max_chars
            chunks.append({'rfc': rfc, 'text': part[:cut].strip()})
            part = part[cut:].strip()
        if len(part) > 100:
            chunks.append({'rfc': rfc, 'text': part})
    return chunks

chunks = []
for n in RFCS:
    with open(f'data/rfc/rfc{n}.txt') as f:
        chunks += chunk_rfc(clean_rfc(f.read()), n)
print(f'{len(chunks)} chunks from {len(RFCS)} RFCs')

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')
emb = embedder.encode([c['text'] for c in chunks],
                      normalize_embeddings=True, show_progress_bar=True)

def retrieve(query, k=5):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = emb @ q  # 余弦相似度：向量已做单位归一化
    return [chunks[i] for i in np.argsort(-scores)[:k]]

def answer_question(question, k=5):
    """检索 top-k 个块，带引用作答。返回 (answer, hits)。"""
    hits = retrieve(question, k)
    context = '\n\n'.join(f"[RFC {h['rfc']}]\n{h['text']}" for h in hits)
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=1024,
        messages=[
            {'role': 'system',
             'content': ('Answer using ONLY the provided RFC excerpts. Cite the RFC number for '
                         'each claim, e.g. (RFC 9110). If the excerpts do not contain the answer, '
                         'say plainly that the corpus does not cover it. Do not guess.')},
            {'role': 'user', 'content': f'{context}\n\nQuestion: {question}'},
        ],
    )
    return resp.choices[0].message.content, hits

## 什么样的黄金集才算好

黄金集是一份你信得过的「问题/参考答案」列表。有四条性质比规模更重要：

- **它覆盖你害怕的失败模式，而不只是顺利路径。** 我们这份混合了简单查找（404）、多部分答案（OAuth 角色），以及关键词出现在多份 RFC 里的问题（幂等方法同时存在于 RFC 2616 和 RFC 9110，检索器会挑哪一个？）。
- **它包含无法回答的问题。** 语料里没有 QUIC、没有 TLS、没有 IMAP。这些问题的正确答案是拒答。一个自信作答它们的 RAG 系统就是在幻觉，你需要一个能抓住它的测试。
- **它小到可以每次变更都跑。** ~15 个 case 只要几分钱、几分钟。一个 500 case 的套件每月跑一次，回归就会晚一个月才被发现。
- **它从生产失败中生长。** 当用户拿到一个坏答案，那个问题和「本该给出的答案」一起进集合。每个 bug 都变成一个测试 case，和回归测试一样的纪律，只是用在模型行为上。

下面每个 case 都带有 `must_mention`（正确答案不能省略的关键词）和 `source_rfc`（检索器必须命中的文档）。正是这两个字段让廉价的确定性检查成为可能。


In [ ]:
# 黄金集：针对真实语料写的 ~15 个 case。每个 case 都固定了
# 正确答案必须包含什么（must_mention），以及检索器应当命中
# 哪份 RFC（source_rfc）。无法回答的 case 用 source_rfc=None —— 正确
# 行为是拒答，而幻觉出一个答案是硬失败。
GOLDEN_SET = [
    {'question': 'How does the TCP three-way handshake establish a connection?',
     'reference_answer': 'The initiator sends a SYN carrying its initial sequence number; '
                         'the peer responds with SYN-ACK carrying its own sequence number and '
                         'acknowledging the first; the initiator replies with an ACK. Both '
                         'sides now have synchronized sequence numbers.',
     'source_rfc': 793, 'must_mention': ['SYN', 'ACK', 'sequence'], 'unanswerable': False},
    {'question': 'What does the HTTP 404 status code mean?',
     'reference_answer': '404 Not Found: the origin server did not find a current '
                         'representation for the target resource, or is not willing to '
                         'disclose that one exists.',
     'source_rfc': 9110, 'must_mention': ['404', 'not found'], 'unanswerable': False},
    {'question': 'What are the four roles defined in the OAuth 2.0 framework?',
     'reference_answer': 'Resource owner, resource server, client, and authorization server.',
     'source_rfc': 6749,
     'must_mention': ['resource owner', 'client', 'authorization server', 'resource server'],
     'unanswerable': False},
    {'question': 'What is the difference between an A record and a CNAME record in DNS?',
     'reference_answer': 'An A record maps a name to a 32-bit IPv4 host address; a CNAME '
                         'record maps an alias to the canonical name of its target.',
     'source_rfc': 1035, 'must_mention': ['address', 'canonical'], 'unanswerable': False},
    {'question': 'How does a client upgrade an HTTP connection to a WebSocket connection?',
     'reference_answer': 'The client sends a GET request with Upgrade: websocket and '
                         'Connection: Upgrade headers plus a Sec-WebSocket-Key; the server '
                         'replies 101 Switching Protocols with a Sec-WebSocket-Accept value '
                         'derived from the key.',
     'source_rfc': 6455, 'must_mention': ['Upgrade', '101', 'Sec-WebSocket-Key'],
     'unanswerable': False},
    {'question': 'Which SMTP commands send a mail message, and in what order?',
     'reference_answer': 'MAIL FROM identifies the sender, one or more RCPT TO commands name '
                         'recipients, then DATA introduces the message text, terminated by a '
                         'line containing only a period.',
     'source_rfc': 5321, 'must_mention': ['MAIL', 'RCPT', 'DATA'], 'unanswerable': False},
    {'question': 'What is the purpose of the Time to Live field in the IPv4 header?',
     'reference_answer': 'TTL is an upper bound on datagram lifetime. Every module that '
                         'processes the datagram decrements it, and the datagram is discarded '
                         'when it reaches zero, so packets cannot loop forever.',
     'source_rfc': 791, 'must_mention': ['zero', 'discard'], 'unanswerable': False},
    {'question': 'What is the AS_PATH attribute in BGP and why does it matter?',
     'reference_answer': 'AS_PATH records the sequence of autonomous systems a route '
                         'advertisement has traversed. BGP uses it for loop detection (a '
                         'router rejects routes containing its own AS number) and in route '
                         'selection.',
     'source_rfc': 4271, 'must_mention': ['autonomous system', 'loop'], 'unanswerable': False},
    {'question': 'How does HTTP/2 carry multiple requests over a single connection?',
     'reference_answer': 'HTTP/2 multiplexes exchanges as independent streams over one '
                         'connection. Each frame carries a stream identifier, so concurrent '
                         'requests and responses interleave without blocking each other.',
     'source_rfc': 7540, 'must_mention': ['stream', 'frame'], 'unanswerable': False},
    {'question': 'Which HTTP methods are defined as idempotent?',
     'reference_answer': 'PUT and DELETE, plus all safe methods: GET, HEAD, OPTIONS, TRACE.',
     'source_rfc': 9110, 'must_mention': ['PUT', 'DELETE', 'GET'], 'unanswerable': False},
    {'question': 'Why does a TCP connection enter the TIME-WAIT state, and for how long?',
     'reference_answer': 'The side that closes actively waits in TIME-WAIT for twice the '
                         'maximum segment lifetime (2*MSL), so delayed segments from the old '
                         'incarnation die off before the socket pair can be reused.',
     'source_rfc': 793, 'must_mention': ['TIME-WAIT', 'MSL'], 'unanswerable': False},
    {'question': 'Walk through the OAuth 2.0 authorization code grant flow.',
     'reference_answer': 'The client redirects the resource owner to the authorization '
                         'server; after authentication and consent the server redirects back '
                         'with an authorization code; the client exchanges the code, with its '
                         'own credentials, at the token endpoint for an access token.',
     'source_rfc': 6749, 'must_mention': ['authorization code', 'access token', 'redirect'],
     'unanswerable': False},
    {'question': 'What does HTTP status 503 indicate, and which header can accompany it?',
     'reference_answer': '503 Service Unavailable: the server is currently unable to handle '
                         'the request, due to overload or maintenance. It may send Retry-After '
                         'to indicate how long to wait.',
     'source_rfc': 9110, 'must_mention': ['503', 'Retry-After'], 'unanswerable': False},
    # --- 刻意设为无法回答：正确行为是拒答 ---
    {'question': 'How does QUIC combine the transport and cryptographic handshakes?',
     'reference_answer': 'Declines: QUIC (RFC 9000) is not in the corpus.',
     'source_rfc': None, 'must_mention': [], 'unanswerable': True},
    {'question': 'What cipher suites does TLS 1.3 define?',
     'reference_answer': 'Declines: TLS 1.3 (RFC 8446) is not in the corpus.',
     'source_rfc': None, 'must_mention': [], 'unanswerable': True},
    {'question': 'Which port does IMAP use, and how does a client select a mailbox?',
     'reference_answer': 'Declines: IMAP is not in the corpus (SMTP is the only mail '
                         'protocol here).',
     'source_rfc': None, 'must_mention': [], 'unanswerable': True},
]
print(len(GOLDEN_SET), 'cases,',
      sum(c['unanswerable'] for c in GOLDEN_SET), 'unanswerable')


## 两个确定性指标，在任何 LLM 评判之前

在花钱请评判模型之前，先把免费且即时的检查能榨出来的东西都榨出来：

1. **检索命中率：** 正确的 RFC 有没有出现在 top-k 里？这把检索器单独隔离出来。如果正确的文档压根没出现，生成器就毫无机会，再怎么调 prompt 也救不回来。
2. **关键词覆盖率：** `must_mention` 里的词有多大比例出现在生成的答案里？一个关于 TCP 握手的答案如果从不提 "SYN"，那就是错的，没得商量，你不需要评判模型也知道。

对于无法回答的 case，我们做一次粗糙的拒答措辞标记扫描。它很脆弱，但没关系。确定性检查脆弱，却免费又快速；它们的职责是抓住严重回归，并分诊出哪些值得送给昂贵的评判模型。先便宜的过滤，后昂贵的打分：刻意沿用和第 03 节里 retrieval→reranking 相同的漏斗逻辑。


In [ ]:
def retrieval_hit(case, hits):
    """正确的 RFC 是否出现在检索到的 top-k 里？无法回答的 case 返回 None。"""
    if case['source_rfc'] is None:
        return None
    return case['source_rfc'] in [h['rfc'] for h in hits]

def keyword_coverage(case, answer):
    """答案中出现的 must_mention 关键词比例（不区分大小写）。"""
    if not case['must_mention']:
        return None
    a = answer.lower()
    return sum(kw.lower() in a for kw in case['must_mention']) / len(case['must_mention'])

DECLINE_MARKERS = ['does not cover', "doesn't cover", 'not covered', 'do not contain',
                   'does not contain', 'not in the corpus', 'cannot answer',
                   'no information', 'not addressed', 'do not include']

def declined(answer):
    """针对无法回答 case 的粗糙拒答检测器。刻意做得脆弱 ——
    下一个 notebook 里的 LLM 评判会正经地做这件事。这里它只需要抓住
    那种严重失败：对语料支撑不了的问题自信作答。
    """
    a = answer.lower()
    return any(m in a for m in DECLINE_MARKERS)


In [ ]:
# 把整个黄金集跑过 RAG 系统。
# 成本提示：每个 case 调一次 MODEL。小到可以
# 每次变更都跑，而这正是把集合保持小的全部意义。
results = []
for case in GOLDEN_SET:
    answer, hits = answer_question(case['question'])
    row = {
        'question': case['question'],
        'answer': answer,
        'hit': retrieval_hit(case, hits),
        'coverage': keyword_coverage(case, answer),
        'declined': declined(answer),
        'unanswerable': case['unanswerable'],
    }
    if case['unanswerable']:
        row['ok'] = row['declined']  # 正确行为只有拒答
    else:
        row['ok'] = bool(row['hit']) and row['coverage'] >= 0.6
    results.append(row)
    print('PASS' if row['ok'] else 'FAIL', '|', case['question'][:65])

In [ ]:
# 结果表 —— 纯 Python，16 行不需要 pandas。
def fmt(v):
    if v is None: return '   -'
    if isinstance(v, bool): return ' yes' if v else '  NO'
    return f'{v:.2f}'

print(f"{'hit':>4}  {'cov':>4}  {'ok':>4}  question")
print('-' * 78)
for r in results:
    print(f"{fmt(r['hit'])}  {fmt(r['coverage'])}  {fmt(r['ok'])}  "
          f"{r['question'][:60]}")

answerable = [r for r in results if not r['unanswerable']]
unanswerable = [r for r in results if r['unanswerable']]
print('-' * 78)
print(f'retrieval hit-rate : {sum(r["hit"] for r in answerable)}/{len(answerable)}')
print(f'mean kw coverage   : '
      f'{sum(r["coverage"] for r in answerable) / len(answerable):.2f}')
print(f'unanswerable OK    : {sum(r["ok"] for r in unanswerable)}/{len(unanswerable)}')


跑一下，注意失败集中在哪里。这份语料上的典型模式：命中率的漏失落在词汇横跨多份 RFC 的问题上（HTTP 问题检索到 RFC 2616 而不是 9110 很常见，因为两者都描述状态码，而余弦相似度分不清「现行规范」和「已废弃规范」）。覆盖率漏失往往是差之毫厘：答案是对的，但关键词的措辞不同，而这正是催生下一个 notebook 里 LLM 评判的脆弱性。

如果你碰到一个无法回答的失败，也去检查一下，读读真实的答案。一个自信、引用齐全、却错误的答案，是 RAG 系统最可怕的失败模式，而现在你有了一个能抓住它的测试。

**它在整个技术栈里的位置：** 确定性检查（本 notebook）→ 负责语义层的 LLM 评判（下一个 notebook）→ 一个能 diff 多次运行并对变更设卡的回归测试骨架（notebook 3）。托管的评测平台和开源框架（promptfoo、Braintrust、LangSmith 之类）把同样的分层打包成带仪表盘、运行历史和 CI 钩子的产品。以后值得采用，但先手写一遍，你才知道它们在自动化什么。


## 练习

1. **加五个难 case。** 写出你预期系统会*失败*的 case：一个答案藏在表格或 ASCII 图里（RFC 793 的状态图）、一个答案横跨两份 RFC 的问题，以及一个用了文中从不出现的同义词（用「web address」指 URI）的问题。跑一遍集合，检验你的预测。
2. **修补拒答检测器的盲区。** 构造一个无法回答的问题，让模型用不在 `DECLINE_MARKERS` 里的词来拒答，从而让 `declined()` 报出假失败。然后把检查反过来：不去检测拒答，而是检测*自信的答案*（有引用、不含糊其辞）并标记它们。哪个方向更不脆弱？
3. **测量命中率对 k 的敏感度。** 对 k = 1, 3, 5, 10 重跑纯检索指标（不需要生成调用，只用 `retrieve()`），并打印每个 k 的命中率。找出能让命中率保持最大的最小 k。
4. **把一个「生产 bug」变成 case。** 向系统问三个集合外的问题，直到找到一个坏答案。为它写出黄金集条目（问题、来自 RFC 原文的参考答案、`must_mention`、`source_rfc`），并确认这个新 case 在修复前失败、在提高 k 或改进分块后通过。
